# Alakoro + DASCore: Integração DASDAE

Este notebook demonstra a conversão bidirecional entre AlakoroPatch e DASCore Patch/Spool, a leitura/escrita de formatos suportados pelo DASCore e o uso do pipeline híbrido DASCore + C++20.

In [ ]:
import numpy as np
import dascore as dc

from src.io.alakoro_spool import AlakoroPatch, AlakoroSpool
from src.io.dasdae import DASDAEAdapter
from src.io.dascore_formats import read, write, supported_formats
from src.processing.hybrid_pipeline import HybridPipeline

## 1. Conversão básica Alakoro ↔ DASCore

In [ ]:
# Criar dados sintéticos
data = np.random.randn(500, 32)
patch = DASDAEAdapter.array_to_patch(data, dt_s=1.0, dx_m=2.0, modality='das')
alakoro = AlakoroPatch(patch, well_id='BRA-001')
alakoro

In [ ]:
# Operações DASCore via AlakoroPatch
detrended = alakoro.detrend()
filtered = detrended.pass_filter((0.5, 25.0))
filtered.shape

In [ ]:
# Spool
spool = AlakoroSpool([alakoro, detrended, filtered])
spool.get_contents()

## 2. Leitura e escrita de formatos DASCore

In [ ]:
# Formatos detectados no DASCore
supported_formats()[:10]

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    # Escrever em formato DASDAE
    path = Path(tmp) / 'example.dasdae'
    write(alakoro, path)
    back = read(path, well_id='BRA-001')
    print(back, back.shape, np.allclose(back.data, data))

    # Escrever em pickle (preserva spools)
    spool_path = Path(tmp) / 'spool.pickle'
    write(spool, spool_path)
    back_spool = read(spool_path, well_id='BRA-001')
    print(back_spool, len(back_spool))

## 3. Pipeline híbrido DASCore + C++20

In [ ]:
pipeline = (
    HybridPipeline(alakoro)
    .dascore('detrend', dim='time', type='linear')
    .dascore('pass_filter', time=(0.5, 25.0))
    .cpp('median_filter_1d', window_size=5)
    .cpp('wavelet_denoise', scales=[1.0, 2.0, 4.0],
         sample_rate_hz=1.0, threshold=0.5)
    .dascore('decimate', time=2)
)
result = pipeline.to_patch()
print(result)
print('Passos:', pipeline.history)

In [ ]:
# Processadores que retornam arrays (psd, cwt, etc.) via apply_array
psd = HybridPipeline(alakoro).apply_array('psd', sample_rate_hz=1.0)
print('PSD shape:', psd.shape)